# When more communication hurts: LPAC and the communication range

Companion notebook of the Medium post *When More Communication Hurts: Debugging a Graph Neural
Network for Robot Swarms* (Stanford CS224W, Autumn 2026). Full study and interactive figures:
[https://loicpoisson.github.io/lpac-range-transfer/](https://loicpoisson.github.io/lpac-range-transfer/).

**What it does, with no training at all:**

1. Runs the published LPAC checkpoint [Agarwal et al., IEEE T-RO 2025] at 256, 512 and 1024 m and
   shows that its coverage cost explodes as the communication range grows.
2. Runs an inference-time factorial at 1024 m: same weights and simulator, change only the GNN's
   graph or only the CNN's neighbour map. The map is the culprit, not the graph.
3. Applies the fix: draw the neighbour map at a fixed 256 m scale. The cost comes back to the
   model's own optimum.
4. Runs V4, our map-less variant with bounded edge features (its checkpoint is downloaded, 7.6 MB).

**Runtime.** `QUICK = True` (default) uses 2 environments with full 600-step episodes and runs them
in parallel on Colab's 2 CPUs: about 35 minutes for parts 1-3 and 15 more for part 4.
`QUICK = False` uses the 11 released environments: several hours. No GPU needed. Every cost is the coverage cost divided by that of a
clairvoyant controller that knows the whole field, averaged over the episode: lower is better,
1.0 is perfect.

In [ ]:
# The simulator ships as a manylinux wheel (Python 3.10-3.13); torch and torchvision are preinstalled on Colab.
import importlib.util, subprocess, sys
for module, spec in [("coverage_control", "coverage_control==1.3.1"), ("torch_geometric", "torch_geometric"),
                     ("torchvision", "torchvision")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])
import torch, torch_geometric, coverage_control
print("torch", torch.__version__, "| PyG", torch_geometric.__version__, "| coverage_control", coverage_control.__version__)

In [ ]:
import os
from pathlib import Path
QUICK = True                     # False: the 11 released environments (several hours)
NUM_ENVS = 2 if QUICK else 11
NUM_STEPS = 600                  # episode length used throughout the study
WORKERS = 2                      # parallel processes, one environment each (Colab CPU runtime has 2 cores)
V4_URL = ("https://raw.githubusercontent.com/loicpoisson/lpac-range-transfer/models/"
          "v4_mixed_ranges_seed0.pt")   # V4 checkpoint (7.6 MB); set to "" to skip part 4
ROOT = Path(os.environ.get("LPAC_NB_ROOT", "/content"))
OUT = ROOT / "lpac_range"
OUT.mkdir(parents=True, exist_ok=True)
os.environ.update(OMP_NUM_THREADS="1", OMP_THREAD_LIMIT="1", OMP_WAIT_POLICY="PASSIVE")

In [ ]:
# Published checkpoint, the 11 evaluation environments and the authors' parameters (release v1.3.1, ~300 MB).
import tarfile, urllib.request
DATA_ROOT = ROOT / "lpac_data"
DATA_DIR = DATA_ROOT / "lpac"
if not (DATA_DIR / "models" / "model_k3_1024_state_dict.pt").is_file():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    archive = DATA_ROOT / "lpac_CoverageControl.tar.gz"
    urllib.request.urlretrieve("https://github.com/KumarRobotics/CoverageControl/releases/download/v1.3.1/"
                               "lpac_CoverageControl.tar.gz", archive)
    with tarfile.open(archive) as tar:
        tar.extractall(DATA_ROOT)
    archive.unlink()
os.environ["LPAC_DATA_DIR"] = str(DATA_DIR)
print(sorted(p.name for p in DATA_DIR.iterdir()))

## The code: package `lpac_study`

The next cells write our study package to the runtime's disk, unchanged from the project
repository. Everything graph-related goes through PyTorch Geometric: the library builds a radius
graph over the robots, the GNN is five `TAGConv` layers (K = 3), and our variants are
`MessagePassing` layers. `neighbourhood.py` rebuilds the neighbour map and the graph so that
either can be changed at inference; `factorial.py` runs those interventions.

In [ ]:
import os
os.makedirs('lpac_study', exist_ok=True)

In [ ]:
%%writefile lpac_study/__init__.py
"""Reproduction study of the LPAC communication-range anomaly (Agarwal et al., T-RO 2025).

The package evaluates the published, pre-trained LPAC-K3 model and the CVT
baselines of the CoverageControl library under varying communication ranges,
and records graph statistics (degree, effective rank of GNN layer outputs)
along the LPAC trajectories.  No training is performed anywhere.
"""

In [ ]:
%%writefile lpac_study/paths.py
"""Locate the files of the published LPAC release archive without hard-coded paths."""
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path

DATA_DIR_ENV_VAR = "LPAC_DATA_DIR"
RELEASE_URL = (
    "https://github.com/KumarRobotics/CoverageControl/releases/download/"
    "v1.3.1/lpac_CoverageControl.tar.gz"
)


@dataclass(frozen=True)
class DataPaths:
    """Paths inside the extracted ``lpac/`` folder of the release archive."""

    root: Path

    @property
    def model_state_dict(self) -> Path:
        return self.root / "models" / "model_k3_1024_state_dict.pt"

    @property
    def learning_params(self) -> Path:
        return self.root / "params" / "learning_params.toml"

    @property
    def env_params(self) -> Path:
        return self.root / "params" / "coverage_control_params.toml"

    @property
    def envs_dir(self) -> Path:
        return self.root / "envs"

    def env_file(self, index: int) -> Path:
        """Importance density field (Gaussian features) of environment ``index``."""
        return self.envs_dir / f"{index}.env"

    def pos_file(self, index: int) -> Path:
        """Initial robot positions of environment ``index`` (one ``x y`` pair per line)."""
        return self.envs_dir / f"{index}.pos"

    def count_environments(self) -> int:
        """Number of environments for which both ``.env`` and ``.pos`` files exist."""
        count = 0
        while self.env_file(count).is_file() and self.pos_file(count).is_file():
            count += 1
        return count

    def check(self) -> None:
        """Raise a clear error when a required file is missing."""
        for path in (self.model_state_dict, self.learning_params, self.env_params):
            if not path.is_file():
                raise FileNotFoundError(f"Missing release file: {path}")
        if self.count_environments() == 0:
            raise FileNotFoundError(f"No environment files found in {self.envs_dir}")


def resolve_data_dir(cli_value: str | None = None) -> DataPaths:
    """Resolve the data directory from the CLI, else from ``$LPAC_DATA_DIR``.

    The directory is the ``lpac/`` folder produced by extracting the release
    archive (it contains ``models/``, ``envs/`` and ``params/``).
    """
    value = cli_value or os.environ.get(DATA_DIR_ENV_VAR)
    if not value:
        raise SystemExit(
            f"Give --data-dir or set {DATA_DIR_ENV_VAR} to the extracted 'lpac' folder."
        )
    paths = DataPaths(Path(value).expanduser().resolve())
    paths.check()
    return paths

In [ ]:
%%writefile lpac_study/simulation.py
"""Thin wrappers around the CoverageControl simulator shared by all experiments.

Everything here is deterministic: the simulator only draws random numbers when
it generates a new environment or when position noise is enabled, and both
are avoided (environments come from the release archive, noise is off).
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import coverage_control as cc
from coverage_control import CoverageSystem, PointVector, WorldIDF


def make_parameters(
    env_params_file: Path, communication_range: float, num_robots: int
) -> cc.Parameters:
    """Load the authors' simulation parameters and override range and team size.

    ``CoverageSystem`` copies the parameters at construction, so the overrides
    must happen before the environment is built.
    """
    params = cc.Parameters(str(env_params_file))
    params.pCommunicationRange = float(communication_range)
    params.pNumRobots = int(num_robots)
    return params


def load_initial_positions(pos_file: Path, num_robots: int) -> np.ndarray:
    """Return the first ``num_robots`` initial positions stored in a ``.pos`` file.

    The released files hold 32 positions.  The simulator refuses a ``.pos``
    file whose length differs from ``NumRobots``, so smaller teams are built by
    keeping the first rows of the file (a fixed, documented, seed-free choice).
    """
    positions = np.loadtxt(pos_file, dtype=np.float64).reshape(-1, 2)
    if num_robots > len(positions):
        raise ValueError(
            f"{pos_file} holds {len(positions)} positions, fewer than {num_robots}"
        )
    return positions[:num_robots]


def make_environment(
    params: cc.Parameters, env_file: Path, initial_positions: np.ndarray
) -> CoverageSystem:
    """Build a fresh coverage environment from an IDF file and initial positions."""
    world_idf = WorldIDF(params, str(env_file))
    return CoverageSystem(params, world_idf, PointVector(initial_positions))


def normalized_cost(env: CoverageSystem, initial_cost: float) -> float:
    """Coverage cost divided by the cost at the start of the episode (as in the paper)."""
    return env.GetObjectiveValue() / initial_cost

In [ ]:
%%writefile lpac_study/graph_stats.py
"""Statistics of the communication graph actually fed to the GNN."""
from __future__ import annotations

from dataclasses import dataclass

import torch
from torch_geometric.utils import degree


@dataclass(frozen=True)
class DegreeStats:
    """Per-step degree statistics of one communication graph."""

    mean: float
    std: float
    isolated_fraction: float


def degree_stats(edge_index: torch.Tensor, num_nodes: int) -> DegreeStats:
    """Mean and standard deviation of node degree from a PyG ``edge_index``.

    The communication graph is symmetric, so in- and out-degree coincide.
    """
    deg = degree(edge_index[1], num_nodes=num_nodes, dtype=torch.float32)
    return DegreeStats(
        mean=float(deg.mean()),
        std=float(deg.std(unbiased=False)),
        isolated_fraction=float((deg == 0).float().mean()),
    )

In [ ]:
%%writefile lpac_study/effective_rank.py
"""Effective rank of a node-representation matrix (Roy & Vetterli, 2007).

Used as the over-smoothing measure recommended by Zhang et al. (ICLR 2026,
arXiv:2502.04591): it collapses towards 1 when all node representations
become collinear, where the Dirichlet energy stays uninformative for shallow
networks.
"""
from __future__ import annotations

import math

import torch


def effective_rank(features: torch.Tensor) -> float:
    """Exponential of the Shannon entropy of the normalized singular values.

    Args:
        features: matrix of shape ``(num_nodes, num_channels)``.

    Returns:
        A value in ``[1, min(num_nodes, num_channels)]``, or ``nan`` when the
        matrix is identically zero.
    """
    singular_values = torch.linalg.svdvals(features.detach().double())
    total = singular_values.sum()
    if total <= 0:
        return math.nan
    probabilities = singular_values / total
    probabilities = probabilities[probabilities > 0]
    entropy = -(probabilities * probabilities.log()).sum()
    return float(torch.exp(entropy))

In [ ]:
%%writefile lpac_study/neighbourhood.py
"""Neighbourhood construction for the inference-time factorial experiment.

The neighbourhood enters LPAC through two inputs that this module rebuilds
from raw robot positions:

* the communication graph given to the five TAGConv layers (``edge_index``);
* the two "communication map" channels given to the CNN, which encode the
  relative position of neighbours on a ``map_size x map_size`` grid whose
  resolution scales with the communication range.

The map reconstruction mirrors ``CoverageEnvUtils.get_communication_maps`` of
coverage_control 1.3.1 (a robot ``j`` is a neighbour of ``i`` when
``|p_j - p_i| < range``; cell index ``round(rel * map_size / (2 range) +
map_size / 2 - 1/2)``; cell value ``rel / range`` summed over the robots that
fall in the same cell).  ``checks.py`` verifies the reconstruction against the
library before any neighbour restriction is applied.
"""
from __future__ import annotations

import torch
from torch_geometric.utils import dense_to_sparse, to_undirected


def robot_positions_double(env) -> torch.Tensor:
    """Raw robot positions in float64, exactly as the C++ simulator holds them.

    The library converts positions to float32 before use; computing relative
    positions in double and casting afterwards reproduces the C++ maps bit for bit.
    """
    return torch.tensor([[p[0], p[1]] for p in env.GetRobotPositions()], dtype=torch.float64)


def relative_positions(positions: torch.Tensor) -> torch.Tensor:
    """``rel[i, j] = p_j - p_i`` for raw positions of shape ``(N, 2)``."""
    return positions.unsqueeze(0) - positions.unsqueeze(1)


def in_range_mask(positions: torch.Tensor, comm_range: float) -> torch.Tensor:
    """Boolean ``(N, N)`` mask of robots strictly within communication range (no self-loops)."""
    distances = torch.cdist(positions, positions)
    mask = distances < comm_range
    mask.fill_diagonal_(False)
    return mask


def knn_within_range_mask(positions: torch.Tensor, comm_range: float, k: int) -> torch.Tensor:
    """Directed mask keeping, for every robot, its ``k`` nearest robots among those in range.

    Robots with fewer than ``k`` neighbours in range keep all of them.  This is
    the k-nearest-neighbour rule of Muthusamy et al. (arXiv:2409.19829)
    restricted to physically reachable robots.
    """
    distances = torch.cdist(positions, positions)
    reachable = in_range_mask(positions, comm_range)
    distances = distances.masked_fill(~reachable, float("inf"))
    k = min(k, positions.shape[0] - 1)
    nearest = distances.topk(k, dim=1, largest=False).indices
    mask = torch.zeros_like(reachable)
    mask.scatter_(1, nearest, True)
    return mask & reachable


def symmetrize(mask: torch.Tensor) -> torch.Tensor:
    """Undirected version of a neighbour mask: keep an edge if either robot selected the other."""
    return mask | mask.T


def mask_to_edge_index(mask: torch.Tensor) -> torch.Tensor:
    """PyG ``edge_index`` (undirected, coalesced) from a symmetric boolean mask."""
    edge_index, _ = dense_to_sparse(mask.to(torch.float32))
    return to_undirected(edge_index, num_nodes=mask.shape[0])


def comm_maps_from_positions(
    positions: torch.Tensor, neighbour_mask: torch.Tensor, comm_range: float, map_size: int,
    clamp: bool = False,
) -> torch.Tensor:
    """Communication maps ``(N, 2, map_size, map_size)`` for the neighbours of ``neighbour_mask``.

    Args:
        positions: raw robot positions, shape ``(N, 2)``, in metres.
        neighbour_mask: ``(N, N)`` boolean, row ``i`` = robots drawn on robot ``i``'s map.
        comm_range: range used for the grid resolution and the value normalization.
        map_size: side of the square map (32 for the published model).
        clamp: project a neighbour farther than ``comm_range`` onto the circle of that radius
            instead of refusing it (same direction, distance ``comm_range``), so it lands on the
            outermost cell of the map in its direction with a value of norm 1. With ``clamp``
            false a neighbour outside the map is an error, which is what every other level of
            the factorial relies on.
    """
    num_robots = positions.shape[0]
    rel = relative_positions(positions).to(torch.float32)  # the library works on float32 relative positions
    if clamp:
        norm = rel.norm(dim=-1, keepdim=True).clamp_min(1e-9)
        rel = rel * torch.clamp(comm_range / norm, max=1.0)
    scaled = torch.round(rel * map_size / (comm_range * 2.0) + (map_size / 2.0 - 0.5)).long()
    values = rel / comm_range
    maps = torch.zeros((num_robots, 2, map_size, map_size), dtype=torch.float32)
    robot, neighbour = neighbour_mask.nonzero(as_tuple=True)
    cell_x, cell_y = scaled[robot, neighbour, 0], scaled[robot, neighbour, 1]
    if clamp:  # rounding of a clamped neighbour can land one cell past the border
        cell_x = cell_x.clamp(0, map_size - 1)
        cell_y = cell_y.clamp(0, map_size - 1)
    elif len(robot) and (cell_x.min() < 0 or cell_y.min() < 0 or cell_x.max() >= map_size or cell_y.max() >= map_size):
        raise ValueError("a neighbour falls outside the communication map")
    for channel in (0, 1):
        maps[:, channel].index_put_((robot, cell_x, cell_y), values[robot, neighbour, channel], accumulate=True)
    return maps

In [ ]:
%%writefile lpac_study/variants.py
"""LPAC variants for the ablation of the neighbour map (phase 2 of the project).

Three models, same CNN, same action MLP, same training recipe:

* ``V0`` the original LPAC: 4-channel CNN input (local map, two neighbour-map channels,
  obstacle map) and five ``TAGConv`` layers with K = 3 (receptive field 15 hops);
* ``V1`` LPAC without the neighbour map: 2-channel CNN input, ``TAGConv`` unchanged.
  Control: what does removing the fragile map alone do?
* ``V2`` V1 plus the neighbour geometry on the graph edges: the GNN is five
  ``GATv2Conv`` layers (4 heads of 64, concatenated to 256, ``edge_dim = 2``) whose
  edge attribute is the relative position ``p_j - p_i`` of the sender, divided by the
  world size (the same constant that normalizes ``data.pos``), never by the range.
  Receptive field 5 hops.
* ``V0R`` the original LPAC with one more GNN input per robot: its own communication range
  divided by the world size (experiment H2, trained on teams with a range drawn per robot).
* ``V3`` V1 plus one ``PointNetConv`` layer in front of the unchanged ``TAGConv`` stack.
* ``V4`` V3 with bounded, range-invariant edge features: the unit vector towards the neighbour
  and ``tanh(d / 256 m)`` instead of the raw relative position.
* ``V5`` the original LPAC with three pairs of neighbour-map channels drawn at fixed scales
  (256, 512, 1024 m) instead of one pair drawn at the communication range: 8 CNN channels.
* ``V6`` V5 plus V4's bounded-edge layer: the multi-scale maps feed the CNN and the bounded
  edge features feed a residual front layer ahead of the unchanged TAGConv stack.
* ``V2R`` / ``V2RL`` the GATv2 of V2 with the usual hygiene (pre-norm residual blocks and V4's
  bounded edge features), without and with the log-degree of each robot as a node feature:
  the diagnosis of why V2 underfits (question Q3 of the teaching assistant).
  The layer reads, for every edge, the relative position of the neighbour divided by a
  fixed constant of 256 m (never by the range, never by the world size), and adds its
  output to the TAGConv input (residual connection). Receptive field 16 hops.

The library's ``LPAC`` class is subclassed, not modified; the edge attributes are
computed in ``forward`` from ``data.pos`` and ``data.edge_index``.
"""
from __future__ import annotations

import copy
from pathlib import Path

import torch
import torch_geometric
from coverage_control.nn import LPAC
from coverage_control.nn.data_loaders.loaders import CNNGNNDataset
from torch_geometric.nn import GATv2Conv, PointNetConv

VARIANTS = ("V0", "V1", "V2", "V3", "V0R", "V4", "V5", "V6", "V2R", "V2RL", "V2B", "V2M")
MULTISCALE_SCALES_M = (256.0, 512.0, 1024.0)  # V5: one pair of map channels per scale
EDGE_DIM = 2
HEADS = 4
# V3 constants. ``data.pos`` is (p + W/2) / W with W the world size (1024 m in every dataset of
# this project): multiplying by WORLD_SIZE_M only undoes that normalization and gives metres
# back. POINTNET_SCALE_M is the one scale the layer ever sees: a neighbour 256 m away reads as
# 1.0 whatever the communication range at training or at test time.
WORLD_SIZE_M = 1024.0
POINTNET_SCALE_M = 256.0
POINTNET_HIDDEN = 256


def uses_comm_map(variant: str) -> bool:
    """Whether the CNN input carries neighbour-map channels (two for V0/V0R, six for V5/V6)."""
    return variant in ("V0", "V0R", "V5", "V6")


def uses_log_degree(variant: str) -> bool:
    """Whether the GNN input carries log(1 + in-degree) as an extra node feature (V2RL)."""
    return variant == "V2RL"


def uses_multiscale_maps(variant: str) -> bool:
    """Whether the neighbour maps are the three fixed-scale pairs of V5 (also V6)."""
    return variant in ("V5", "V6")


def uses_range_input(variant: str) -> bool:
    """Whether the GNN input carries each robot's own communication range (``data.ranges``)."""
    return variant == "V0R"


class LPACRangeInput(LPAC):
    """V0R: LPAC whose GNN also reads each robot's own communication range.

    The maps and the graph are those of the original model; the only change is one extra
    input per node, ``r_i / WORLD_SIZE_M`` (0.125 for 128 m, 1 for 1024 m), concatenated to the
    CNN latent and the position before the first TAGConv layer (35 inputs instead of 34).
    Trained on teams where every robot draws its range in {128, 256, 512, 1024} m, the model
    is told at which scale its own map is drawn, which the original model has to guess.
    ``data.ranges`` (one value per node, in metres) is set by the dataset class and by the
    policies; the TAGConv stack, the action MLP and the training recipe are unchanged.
    """

    def __init__(self, config: dict):
        super().__init__(variant_config(config, "V0R"))
        from coverage_control.nn.models.gnn_backbone import GNNBackBone  # the library's own stack, one input wider
        self.gnn_backbone = GNNBackBone(self.config, self.cnn_backbone.latent_size + 3)

    def forward(self, data: torch_geometric.data.Data) -> torch.Tensor:
        x, edge_index, pos = data.x, data.edge_index, data.pos
        ranges = (data.ranges / WORLD_SIZE_M).to(x.dtype).view(-1, 1)
        cnn_output = self.cnn_backbone(x.view(-1, x.shape[-3], x.shape[-2], x.shape[-1]))
        gnn_input = torch.cat([cnn_output, pos, ranges], dim=-1)
        return self.output_linear(self.gnn_mlp(self.gnn_backbone(gnn_input, edge_index)))


class RangeDataset(CNNGNNDataset):
    """``CNNGNNDataset`` that also attaches ``data.ranges`` from ``robot_ranges.pt`` (metres per robot).

    A dataset without that file (generated at a single range) gets the constant ``default_range``.
    """

    def __init__(self, data_dir, stage, use_comm_map, world_size, default_range: float | None = None):
        super().__init__(data_dir, stage, use_comm_map, world_size)
        path = Path(f"{data_dir}/{stage}/robot_ranges.pt")
        if path.is_file():
            self.robot_ranges = torch.load(path, weights_only=True).to(torch.float32)
        else:
            if default_range is None:
                raise FileNotFoundError(f"{path} is missing and no default range was given")
            self.robot_ranges = torch.full((len(self), self.robot_positions.shape[1]), float(default_range))

    def get(self, idx):
        data, targets = super().get(idx)
        data.ranges = self.robot_ranges[idx]
        return data, targets


class MultiScaleDataset(CNNGNNDataset):
    """Dataset for V5 that keeps the six neighbour-map channels sparse until a sample is requested.

    The library's loader densifies ``comm_maps.pt`` at load time; with six channels that is
    100,000 x 32 x 6 x 32 x 32 floats, 78 GB, which does not fit next to the other tensors.
    The maps hold at most a few tens of nonzero cells per robot, so they are kept as the
    sparse COO tensor written by ``redraw_ranges.py --multiscale`` and densified per sample in
    ``get``; the resulting input is identical to what ``DataLoaderUtils.load_maps`` would build
    (local map, six neighbour channels, obstacle map).
    """

    def __init__(self, data_dir, stage, use_comm_map=True, world_size=1024):
        super().__init__(data_dir, stage, False, world_size)  # local and obstacle maps, dense
        comm = torch.load(f"{data_dir}/{stage}/comm_maps.pt", weights_only=True)
        comm = comm.coalesce() if comm.is_sparse else comm.to_sparse_coo().coalesce()
        self.num_comm_channels = comm.shape[2]
        self.comm_shape = tuple(comm.shape[1:])
        # Indexing a coalesced COO tensor by sample scans all of its indices (190 M entries for the
        # 100k mixed set: about 0.3 s per sample, 20 min per epoch on 27 workers). The entries are
        # sorted by sample, so they are split once into per-sample index/value slices.
        indices, values = comm.indices(), comm.values()
        counts = torch.bincount(indices[0], minlength=comm.shape[0]).tolist()
        self.comm_indices = list(torch.split(indices[1:], counts, dim=1))
        self.comm_values = list(torch.split(values, counts))

    def comm_map(self, idx: int) -> torch.Tensor:
        """Dense ``(robots, channels, S, S)`` neighbour maps of one sample."""
        return torch.sparse_coo_tensor(self.comm_indices[idx], self.comm_values[idx], self.comm_shape).to_dense()

    def get(self, idx):
        from coverage_control.nn.data_loaders.data_loader_utils import DataLoaderUtils
        base = self.maps[idx]  # (robots, 2, S, S): local map, obstacle map
        comm = self.comm_map(idx)  # (robots, 6, S, S)
        maps = torch.cat([base[:, :1], comm, base[:, 1:2]], dim=1)
        data = DataLoaderUtils.to_torch_geometric_data(maps, self.edge_weights[idx], self.robot_positions[idx])
        targets = self.targets[idx]
        if targets.dim == 3:
            targets = targets.view(-1, targets.shape[-1])
        return data, targets


class LPACPointNet(LPAC):
    """V3: LPAC without neighbour maps, with one PointNetConv layer ahead of the TAGConv stack.

    Why this layer. In V1 the GNN only knows *that* two robots are linked (binary adjacency)
    plus each robot's absolute position; the relative geometry that the neighbour map used to
    rasterize is gone. ``PointNetConv`` (Qi et al., 2017, as implemented in PyG) is the simplest
    layer that reads geometry on edges: for every edge j -> i it feeds ``[x_j, p_j - p_i]`` to a
    small MLP (``local_nn``), takes the element-wise max over the neighbours of i, and maps the
    result through a second MLP (``global_nn``). Unlike V2, which replaced the whole GNN by GATv2
    attention, V3 keeps the five TAGConv layers exactly as in V1/V0 and only *adds* geometry in
    front of them.

    Design choices, each one deliberate:
    * positions are given to the layer in units of ``POINTNET_SCALE_M`` (256 m), a constant:
      the layer sees the same numbers for the same geometry at every communication range, so
      nothing in its input scales with the range (the flaw of the rasterized map);
    * ``add_self_loops=True`` (PyG default): a robot with no neighbour in range (31 % of the
      robots at 128 m) still gets one message, from itself with relative position 0, instead
      of an undefined max over an empty set;
    * max aggregation (PyG default): invariant to the number of neighbours, which varies from
      ~2 at 128 m to ~30 at 1024 m;
    * residual connection ``h = x + relu(pointnet(x))``: the V1 pathway is left intact, so the
      model can at worst ignore the layer, and the training-loss control against V1 is fair;
    * output width equal to the TAGConv input width (CNN latent 32 + position 2 = 34) so that
      the five TAGConv layers are unchanged, hyperparameters included.
    """

    def __init__(self, config: dict):
        super().__init__(variant_config(config, "V3"))
        width = self.cnn_backbone.latent_size + 2  # 34: what the first TAGConv layer expects
        self.pointnet = PointNetConv(
            local_nn=torch.nn.Sequential(torch.nn.Linear(width + 2, POINTNET_HIDDEN), torch.nn.ReLU(),
                                         torch.nn.Linear(POINTNET_HIDDEN, POINTNET_HIDDEN)),
            global_nn=torch.nn.Sequential(torch.nn.ReLU(), torch.nn.Linear(POINTNET_HIDDEN, width)),
            add_self_loops=True)

    @staticmethod
    def scaled_positions(pos: torch.Tensor) -> torch.Tensor:
        """Positions in units of 256 m: ``PointNetConv`` subtracts them along each edge."""
        return pos * (WORLD_SIZE_M / POINTNET_SCALE_M)

    def forward(self, data: torch_geometric.data.Data) -> torch.Tensor:
        x, edge_index, pos = data.x, data.edge_index, data.pos
        cnn_output = self.cnn_backbone(x.view(-1, x.shape[-3], x.shape[-2], x.shape[-1]))
        gnn_input = torch.cat([cnn_output, pos], dim=-1)  # exactly V1's GNN input
        geometry = torch.relu(self.pointnet(gnn_input, self.scaled_positions(pos), edge_index))
        gnn_output = self.gnn_backbone(gnn_input + geometry, edge_index)  # the unchanged TAGConv stack
        return self.output_linear(self.gnn_mlp(gnn_output))


def variant_config(config: dict, variant: str) -> dict:
    """Copy of the release ``learning_params`` adapted to the variant (2 CNN channels for V1/V2)."""
    config = copy.deepcopy(config)
    if not uses_comm_map(variant):
        config["CNNBackBone"]["InputDim"] = 2
        config["ModelConfig"]["UseCommMaps"] = False
    if uses_multiscale_maps(variant):
        config["CNNBackBone"]["InputDim"] = 2 + 2 * len(MULTISCALE_SCALES_M)
    return config


class EdgeGeometryGNN(torch.nn.Module):
    """Five GATv2 layers reading the relative position of the sender on every edge."""

    def __init__(self, input_dim: int, latent_size: int, num_layers: int, heads: int = HEADS,
                 edge_dim: int = EDGE_DIM):
        super().__init__()
        assert latent_size % heads == 0, "latent size must split evenly across heads"
        per_head = latent_size // heads
        self.num_layers = num_layers  # same attribute as the library's GNNBackBone
        self.layers = torch.nn.ModuleList()
        in_dim = input_dim
        for _ in range(num_layers):
            self.layers.append(GATv2Conv(in_dim, per_head, heads=heads, edge_dim=edge_dim, concat=True))
            in_dim = latent_size

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = torch.relu(layer(x, edge_index, edge_attr))
        return x


class MaxEdgeConv(torch_geometric.nn.MessagePassing):
    """GATv2's message with the softmax attention replaced by a maximum over the neighbours.

    GATv2 scores an edge j -> i from ``s_ij = LeakyReLU(W_l x_i + W_r x_j + W_e e_ij)`` and returns
    the attention-weighted *mean* ``sum_j softmax_j(a . s_ij) W_r x_j``. Here the same three linear
    maps are kept and the layer returns ``max_j s_ij``, coordinate by coordinate. What changes is
    therefore the aggregation alone: a weighted mean is bounded by its inputs and cannot tell one
    neighbour from thirty identical ones, while a maximum keeps the strongest response whatever
    the number of neighbours, which is what the PointNet layer of V3 and V4 does.

    Every robot also receives its own message with a zero edge feature (the self-loop convention
    of V4), so an isolated robot still gets a defined output.
    """

    def __init__(self, in_channels: int, out_channels: int, edge_dim: int = 3, negative_slope: float = 0.2):
        super().__init__(aggr="max", node_dim=0)
        self.lin_l = torch.nn.Linear(in_channels, out_channels)
        self.lin_r = torch.nn.Linear(in_channels, out_channels, bias=False)
        self.lin_edge = torch.nn.Linear(edge_dim, out_channels, bias=False)
        self.negative_slope = negative_slope

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        loops = torch.arange(x.shape[0], device=x.device)
        edge_index = torch.cat([edge_index, torch.stack([loops, loops])], dim=1)
        edge_attr = torch.cat([edge_attr, edge_attr.new_zeros((x.shape[0], edge_attr.shape[1]))], dim=0)
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i: torch.Tensor, x_j: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        return torch.nn.functional.leaky_relu(self.lin_l(x_i) + self.lin_r(x_j) + self.lin_edge(edge_attr), self.negative_slope)


class ResidualEdgeGeometryGNN(torch.nn.Module):
    """GATv2 layers in pre-norm residual blocks: ``x <- x + relu(GATv2(LayerNorm(x)))``.

    The stack of V2 applies ``relu(GATv2(x))`` with no residual and no normalization, so five
    attention layers have to carry the signal on their own; with 15 layers (to match the 15 hops
    of five K = 3 TAGConv layers) that stack does not train at all. The blocks here are the
    standard remedy and leave the attention itself untouched.
    """

    def __init__(self, input_dim: int, latent_size: int, num_layers: int, heads: int = HEADS, edge_dim: int = 3,
                 aggregation: str = "attention"):
        super().__init__()
        assert latent_size % heads == 0, "latent size must split evenly across heads"
        assert aggregation in ("attention", "max")
        per_head = latent_size // heads
        self.num_layers = num_layers  # same attribute as the library's GNNBackBone
        self.aggregation = aggregation
        self.input_proj = torch.nn.Linear(input_dim, latent_size)
        if aggregation == "max":  # V2M: the same blocks around MaxEdgeConv
            self.layers = torch.nn.ModuleList(MaxEdgeConv(latent_size, latent_size, edge_dim) for _ in range(num_layers))
        else:
            self.layers = torch.nn.ModuleList(
                GATv2Conv(latent_size, per_head, heads=heads, edge_dim=edge_dim, concat=True) for _ in range(num_layers))
        self.norms = torch.nn.ModuleList(torch.nn.LayerNorm(latent_size) for _ in range(num_layers))

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)
        for layer, norm in zip(self.layers, self.norms):
            x = x + torch.relu(layer(norm(x), edge_index, edge_attr))
        return x


class LPACResidualEdgeGeometry(LPAC):
    """V2R / V2RL: the GATv2 model of V2 with residual blocks, LayerNorm and bounded edge features.

    Three changes to V2, each one a candidate explanation of its underfitting (train loss 0.62
    after 100 epochs against 0.21 for V0):
    * the edge input is V4's ``[(p_j - p_i) / d, tanh(d / 256 m)]`` instead of ``(p_j - p_i) / W``,
      whose values only span +-0.125 at the 128 m training range;
    * every layer is a pre-norm residual block, so depth does not have to be crossed in one pass;
    * ``V2RL`` adds ``log(1 + in-degree) / log(33)`` to the node input: softmax attention is a
      weighted *mean* and loses the number of neighbours, which the normalized sum of TAGConv
      partly keeps; if that is what V2 misses, the feature restores it.
    The number of layers is read from the config, so the 15-layer run (the receptive field of
    five K = 3 TAGConv layers) is the same class with ``NumLayers = 15``.

    ``V2M`` is V2R with one change, the aggregation: ``MaxEdgeConv`` in place of GATv2, same blocks,
    same LayerNorm, same edge features. V2res matches the map-less TAGConv model at 256 m but still
    degrades with the range (+0.85 from 256 to 1024 m, journal 16.4); the milestone's hypothesis is
    that a softmax-weighted mean is the reason, since V4's maximum does not degrade. V2M tests it.
    """

    def __init__(self, config: dict, variant: str = "V2R"):
        super().__init__(variant_config(config, variant))
        self.log_degree = uses_log_degree(variant)
        width = self.cnn_backbone.latent_size + 2 + (1 if self.log_degree else 0)
        self.gnn_backbone = ResidualEdgeGeometryGNN(width, self.latent_size, self.num_layers,
                                                    aggregation="max" if variant == "V2M" else "attention")

    def forward(self, data: torch_geometric.data.Data) -> torch.Tensor:
        x, edge_index, pos = data.x, data.edge_index, data.pos
        cnn_output = self.cnn_backbone(x.view(-1, x.shape[-3], x.shape[-2], x.shape[-1]))
        parts = [cnn_output, pos]
        if self.log_degree:
            degree = torch.zeros(x.shape[0], device=x.device, dtype=cnn_output.dtype)
            degree.index_add_(0, edge_index[1], torch.ones(edge_index.shape[1], device=x.device, dtype=cnn_output.dtype))
            parts.append((torch.log1p(degree) / float(torch.log(torch.tensor(33.0)))).view(-1, 1))
        gnn_input = torch.cat(parts, dim=-1)
        edge_attr = LPACBoundedGeometry.edge_features(pos, edge_index)
        return self.output_linear(self.gnn_mlp(self.gnn_backbone(gnn_input, edge_index, edge_attr)))


class LPACEdgeGeometry(LPAC):
    """LPAC without neighbour maps, with ``p_j - p_i`` (normalized) as edge attribute of a GATv2 GNN."""

    def __init__(self, config: dict):
        super().__init__(variant_config(config, "V2"))
        self.gnn_backbone = EdgeGeometryGNN(self.cnn_backbone.latent_size + 2, self.latent_size, self.num_layers)

    @staticmethod
    def edge_attributes(pos: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """Relative position of the sender ``j`` seen from the receiver ``i`` for every edge ``(j, i)``.

        ``data.pos`` is already ``(p + W/2) / W`` (world size ``W``), so the difference is
        ``(p_j - p_i) / W``: a fixed constant, independent of the communication range.
        """
        sender, receiver = edge_index[0], edge_index[1]
        return pos[sender] - pos[receiver]

    def forward(self, data: torch_geometric.data.Data) -> torch.Tensor:
        x, edge_index, pos = data.x, data.edge_index, data.pos
        cnn_output = self.cnn_backbone(x.view(-1, x.shape[-3], x.shape[-2], x.shape[-1]))
        gnn_input = torch.cat([cnn_output, pos], dim=-1)
        gnn_output = self.gnn_backbone(gnn_input, edge_index, self.edge_attributes(pos, edge_index))
        return self.output_linear(self.gnn_mlp(gnn_output))


class LPACBoundedEdgeGATv2(LPACEdgeGeometry):
    """V2B: exactly V2, with the bounded edge features of V4 and nothing else changed.

    The single-cause ablation of question Q3. V2res changes three things at once (pre-norm
    residual blocks, LayerNorm, and the edge input), so it identifies a bundle. This variant
    changes only the edge input, from V2's ``(p_j - p_i) / W``, which spans +-0.125 at the 128 m
    training range, to V4's ``[(p_j - p_i) / d, tanh(d / 256 m)]``. The attention stack stays
    plain: ``relu(GATv2(x))``, no residual, no normalization. If the training loss drops most of
    the way from 0.676 towards 0.432, the edge encoding carries the bundle; if it does not, the
    residual blocks do.
    """

    def __init__(self, config: dict):
        LPAC.__init__(self, variant_config(config, "V2"))
        self.gnn_backbone = EdgeGeometryGNN(self.cnn_backbone.latent_size + 2, self.latent_size,
                                            self.num_layers, edge_dim=3)

    def forward(self, data: torch_geometric.data.Data) -> torch.Tensor:
        x, edge_index, pos = data.x, data.edge_index, data.pos
        cnn_output = self.cnn_backbone(x.view(-1, x.shape[-3], x.shape[-2], x.shape[-1]))
        gnn_input = torch.cat([cnn_output, pos], dim=-1)
        edge_attr = LPACBoundedGeometry.edge_features(pos, edge_index)
        return self.output_linear(self.gnn_mlp(self.gnn_backbone(gnn_input, edge_index, edge_attr)))


class BoundedGeometryConv(torch_geometric.nn.MessagePassing):
    """PointNet-style layer whose edge input is bounded and does not scale with the range (V4).

    For every edge j -> i the message is ``local_nn([x_j, e_ij])`` with
    ``e_ij = [(p_j - p_i) / d, tanh(d / scale)]`` (unit direction and a squashed distance, three
    numbers in [-1, 1]); messages are max-aggregated over the neighbours of i (self-loop with
    ``e = 0`` included, so an isolated robot still gets one message) and mapped through
    ``global_nn``. The same structure as ``PointNetConv`` (Qi et al., 2017), with the raw
    relative position replaced by its bounded encoding.
    """

    def __init__(self, local_nn: torch.nn.Module, global_nn: torch.nn.Module):
        super().__init__(aggr="max")
        self.local_nn = local_nn
        self.global_nn = global_nn

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        loops = torch.arange(x.shape[0], device=x.device)
        edge_index = torch.cat([edge_index, torch.stack([loops, loops])], dim=1)
        edge_attr = torch.cat([edge_attr, torch.zeros(x.shape[0], edge_attr.shape[1], device=x.device, dtype=edge_attr.dtype)])
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        return self.global_nn(out)

    def message(self, x_j: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        return self.local_nn(torch.cat([x_j, edge_attr], dim=-1))


def bounded_geometry_layer(width: int) -> "BoundedGeometryConv":
    """The front layer of V4 (and V6): ``local_nn`` on ``[x_j, e_ij]`` (width + 3 inputs), ``global_nn`` back to ``width``."""
    return BoundedGeometryConv(
        local_nn=torch.nn.Sequential(torch.nn.Linear(width + 3, POINTNET_HIDDEN), torch.nn.ReLU(),
                                     torch.nn.Linear(POINTNET_HIDDEN, POINTNET_HIDDEN)),
        global_nn=torch.nn.Sequential(torch.nn.ReLU(), torch.nn.Linear(POINTNET_HIDDEN, width)))


class LPACBoundedGeometry(LPAC):
    """V4: V3 with bounded, range-invariant edge features.

    V3 reads ``(p_j - p_i) / 256 m`` on every edge: at 1024 m those inputs are four times larger
    than at training, and section 13 shows that one recipe for both regimes fits neither. V4
    keeps everything of V3 (no neighbour map, the same five TAGConv layers, the same residual
    front layer, the same hidden width) and only changes what the front layer reads:
    ``e_ij = [(p_j - p_i) / d, tanh(d / 256 m)]``, the unit direction towards the neighbour and
    a squashed distance. Every component lies in [-1, 1] whatever the range; scaling all
    positions by a factor leaves the direction unchanged and only moves the tanh, so a long
    range changes the *number* of edges the layer sees, not the range of its inputs. V3 against
    V4 therefore isolates the edge encoding.
    """

    def __init__(self, config: dict, variant: str = "V4"):
        super().__init__(variant_config(config, variant))
        width = self.cnn_backbone.latent_size + 2  # 34, what the first TAGConv layer expects
        self.geometry = bounded_geometry_layer(width)

    @staticmethod
    def edge_features(pos: torch.Tensor, edge_index: torch.Tensor, scale_m: float = POINTNET_SCALE_M) -> torch.Tensor:
        """``[(p_j - p_i) / d, tanh(d / scale)]`` per edge j -> i, from the world-normalized ``data.pos``."""
        sender, receiver = edge_index[0], edge_index[1]
        rel = (pos[sender] - pos[receiver]) * WORLD_SIZE_M  # metres
        dist = rel.norm(dim=-1, keepdim=True)
        direction = rel / dist.clamp_min(1e-6)
        return torch.cat([direction, torch.tanh(dist / scale_m)], dim=-1)

    def forward(self, data: torch_geometric.data.Data) -> torch.Tensor:
        x, edge_index, pos = data.x, data.edge_index, data.pos
        cnn_output = self.cnn_backbone(x.view(-1, x.shape[-3], x.shape[-2], x.shape[-1]))
        gnn_input = torch.cat([cnn_output, pos], dim=-1)
        geometry = torch.relu(self.geometry(gnn_input, edge_index, self.edge_features(pos, edge_index)))
        gnn_output = self.gnn_backbone(gnn_input + geometry, edge_index)
        return self.output_linear(self.gnn_mlp(gnn_output))


class LPACMultiScaleGeometry(LPACBoundedGeometry):
    """V6: V5's multi-scale maps and V4's bounded edge layer in one model.

    Motivation (section 14 of the journal): V5, whose CNN reads three pairs of neighbour maps at
    fixed scales, holds the 256 m level across ranges but does not improve beyond it; V4, whose
    front layer reads bounded edge features, is the one model whose cost keeps falling with the
    range but it pays 0.14 at 256 m, where it has no map. V6 gives the CNN the V5 input (8
    channels, built exactly as for V5) and the GNN the V4 front layer (same class, same width,
    residual, self-loops), with the TAGConv stack unchanged, to test whether the two gains add.
    Parameters: V5 plus the V4 layer.
    """

    def __init__(self, config: dict):
        super().__init__(config, variant="V6")


class LPACMultiScale(LPAC):
    """V5: LPAC whose neighbour maps are three pairs of channels at fixed scales (256, 512, 1024 m).

    The original model draws one pair of channels at the communication range, so the drawing
    scale moves with the range and the CNN reads an image it was not trained on (section 4).
    The inference-time fix draws one pair at a fixed 256 m scale (section 9) and loses the
    neighbours beyond 256 m. V5 gives the CNN all three: pair k holds the robots within
    min(range, scale_k) drawn at scale_k with the library's value convention (p_j - p_i) /
    scale_k, so at 128 m the three pairs show the same (few) neighbours at three zooms, and at
    1024 m the 256 m pair is the fixed-scale map of section 9 while the 1024 m pair is the
    original map. CNN input 8 channels (local map, six neighbour channels, obstacle map);
    nothing else changes. The datasets are redrawn from the stored positions
    (``training/redraw_ranges.py --multiscale``).
    """

    def __init__(self, config: dict):
        super().__init__(variant_config(config, "V5"))


def build_model(variant: str, config: dict) -> LPAC:
    """The model of a variant from the release ``learning_params`` dictionary."""
    if variant not in VARIANTS:
        raise ValueError(f"unknown variant {variant!r}, expected one of {VARIANTS}")
    if variant == "V2":
        return LPACEdgeGeometry(config)
    if variant == "V3":
        return LPACPointNet(config)
    if variant == "V0R":
        return LPACRangeInput(config)
    if variant == "V4":
        return LPACBoundedGeometry(config)
    if variant == "V5":
        return LPACMultiScale(config)
    if variant == "V6":
        return LPACMultiScaleGeometry(config)
    if variant in ("V2R", "V2RL", "V2M"):
        return LPACResidualEdgeGeometry(config, variant)
    if variant == "V2B":
        return LPACBoundedEdgeGATv2(config)
    return LPAC(variant_config(config, variant))


def parameter_count(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def receptive_field_hops(variant: str, config: dict) -> int:
    """Graph hops seen by a node: layers x K for TAGConv, layers for GATv2, plus one for V3's front layer."""
    layers = config["GNNBackBone"]["NumLayers"]
    if variant in ("V2", "V2R", "V2RL", "V2B", "V2M"):
        return layers
    return layers * config["GNNBackBone"]["NumHops"] + (1 if variant in ("V3", "V4", "V6") else 0)

In [ ]:
%%writefile lpac_study/lpac_policy.py
"""The published LPAC-K3 policy, instrumented to expose its communication graph.

The wrapper mirrors ``coverage_control.algorithms.ControllerNN`` (v1.3.1) but
keeps the PyG ``Data`` object fed to the network and captures the output of
each of the five ``TAGConv`` layers through forward hooks.  ``torch.compile``
is not used so that hooks and CPU execution stay simple and deterministic.
"""
from __future__ import annotations

from pathlib import Path

import torch
import coverage_control as cc
import coverage_control.nn as cc_nn
from coverage_control import CoverageEnvUtils, CoverageSystem, IOUtils, PointVector
from torch_geometric.data import Data

from .paths import DataPaths
from .neighbourhood import comm_maps_from_positions, in_range_mask, robot_positions_double
from .variants import MULTISCALE_SCALES_M, build_model, uses_comm_map, uses_multiscale_maps, uses_range_input

# Controller options of the authors' eval.toml ([[Controllers]] lpac entry).
USE_CNN = True
USE_COMM_MAP = True
CNN_MAP_SIZE = 32


def multiscale_channels(x: torch.Tensor, positions: torch.Tensor, comm_range: float,
                        scales=MULTISCALE_SCALES_M) -> torch.Tensor:
    """V5 input: local map, one pair of neighbour channels per fixed scale, obstacle map.

    ``x`` is the library's 4-channel input (local map, two neighbour channels at the range,
    obstacle map); pair k holds the robots within min(comm_range, scale_k) at scale_k.
    """
    pairs = [comm_maps_from_positions(positions, in_range_mask(positions, min(comm_range, s)), s, CNN_MAP_SIZE).to(x.device)
             for s in scales]
    return torch.cat([x[:, :1], *pairs, x[:, 3:4]], dim=1)


class LPACPolicy:
    """Pre-trained LPAC model with access to its graph and layer activations."""

    def __init__(self, paths: DataPaths, device: torch.device, state_dict: str | Path | None = None,
                 variant: str = "V0"):
        """``state_dict`` overrides the published checkpoint; ``variant`` selects the architecture (V0 = LPAC)."""
        learning_params = IOUtils.load_toml(str(paths.learning_params))
        self.device = device
        self.variant = variant
        self.use_comm_map = USE_COMM_MAP and uses_comm_map(variant)
        self.model = build_model(variant, learning_params).to(device)
        if state_dict is None:
            self.model.load_model(str(paths.model_state_dict))
        else:  # a checkpoint saved on a GPU must be mapped to the evaluation device
            weights = torch.load(str(state_dict), map_location=device, weights_only=True)
            self.model.load_state_dict(weights, strict=True)
        self.model.eval()
        self.actions_mean = self.model.actions_mean.to(device)
        self.actions_std = self.model.actions_std.to(device)
        self.num_layers = self.model.gnn_backbone.num_layers
        self._layer_outputs: list[torch.Tensor] = []
        self._register_layer_hooks()

    def _register_layer_hooks(self) -> None:
        """Capture the post-ReLU output of every TAGConv layer of the GNN backbone."""
        backbone = self.model.gnn_backbone
        layers = (list(backbone.layers) if hasattr(backbone, "layers")
                  else [backbone._modules[f"graph_conv_{i}"] for i in range(self.num_layers)])
        for layer in layers:
            layer.register_forward_hook(self._store_layer_output)

    def _store_layer_output(self, module, inputs, output) -> None:
        # GNNBackBone.forward applies torch.relu after each TAGConv call.
        self._layer_outputs.append(torch.relu(output).detach())

    def build_graph(self, env: CoverageSystem, params: cc.Parameters) -> Data:
        """PyG graph exactly as the library builds it for the controller.

        Node features are the four CNN input channels, ``pos`` the normalized
        positions, ``edge_index`` the symmetric radius graph at the current
        communication range.  ``edge_weight`` is present in the object but the
        LPAC forward pass does not pass it to the GNN, so the adjacency is binary.
        """
        data = CoverageEnvUtils.get_torch_geometric_data(
            env, params, USE_CNN, self.use_comm_map, CNN_MAP_SIZE
        ).to(self.device)
        if uses_range_input(self.variant):  # homogeneous team: every robot has the test range
            data.ranges = torch.full((data.x.shape[0],), float(params.pCommunicationRange), device=self.device)
        if uses_multiscale_maps(self.variant):
            data.x = multiscale_channels(data.x, robot_positions_double(env), float(params.pCommunicationRange)).to(self.device)
        return data

    @torch.no_grad()
    def act(self, data: Data) -> tuple[PointVector, bool, list[torch.Tensor]]:
        """Velocity actions for the current graph, plus convergence flag and layer outputs."""
        self._layer_outputs = []
        actions = self.model(data) * self.actions_std + self.actions_mean
        converged = bool(torch.allclose(actions, torch.zeros_like(actions), atol=1e-5))
        layer_outputs = list(self._layer_outputs)
        return PointVector(actions.cpu().numpy()), converged, layer_outputs

In [ ]:
%%writefile lpac_study/episode.py
"""Run one controller for one episode and record costs and graph statistics."""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import numpy as np
import coverage_control as cc
from coverage_control import CoverageSystem
from coverage_control.algorithms import CentralizedCVT, ClairvoyantCVT, DecentralizedCVT

from .effective_rank import effective_rank
from .graph_stats import degree_stats
from .lpac_policy import LPACPolicy

CVT_ALGORITHMS = {
    "DecentralizedCVT": DecentralizedCVT,
    "CentralizedCVT": CentralizedCVT,
    "ClairvoyantCVT": ClairvoyantCVT,
}
LPAC_NAME = "lpac"
CONTROLLERS = (LPAC_NAME, *CVT_ALGORITHMS)


@dataclass
class EpisodeRecord:
    """Everything measured during one episode of one controller."""

    costs: np.ndarray  # cost / initial cost, index 0 is the initial state
    converged_step: int | None = None
    degree_mean: float = math.nan
    degree_std: float = math.nan
    isolated_fraction: float = math.nan
    layer_eranks: np.ndarray | None = None  # mean effective rank per GNN layer
    layer_eranks_std: np.ndarray | None = None
    _degrees: list = field(default_factory=list, repr=False)

    @property
    def cost_mean(self) -> float:
        """Average cost over the whole episode (the paper's 'average cost')."""
        return float(self.costs.mean())

    @property
    def cost_final(self) -> float:
        return float(self.costs[-1])


def _fill_after_convergence(costs: np.ndarray, step: int, value: float) -> None:
    """Freeze the cost curve once a controller stops moving, as the library evaluator does."""
    costs[step:] = value


def run_cvt_episode(
    algorithm_name: str, params: cc.Parameters, env: CoverageSystem, num_steps: int
) -> EpisodeRecord:
    """Run a CVT baseline (clairvoyant, centralized or decentralized)."""
    algorithm = CVT_ALGORITHMS[algorithm_name](params, env)
    initial_cost = env.GetObjectiveValue()
    costs = np.ones(num_steps)
    record = EpisodeRecord(costs=costs)
    for step in range(1, num_steps):
        algorithm.ComputeActions()
        if env.StepActions(algorithm.GetActions()):
            raise RuntimeError("Simulator reported an error while stepping")
        costs[step] = env.GetObjectiveValue() / initial_cost
        if algorithm.IsConverged():
            _fill_after_convergence(costs, step, costs[step])
            record.converged_step = step
            break
    return record


def run_lpac_episode(
    policy: LPACPolicy,
    params: cc.Parameters,
    env: CoverageSystem,
    num_steps: int,
    record_rank: bool = True,
) -> EpisodeRecord:
    """Run the LPAC policy and measure degree and effective rank at every step."""
    initial_cost = env.GetObjectiveValue()
    num_robots = env.GetNumRobots()
    costs = np.ones(num_steps)
    record = EpisodeRecord(costs=costs)
    degree_sum = degree_sq_sum = isolated_sum = 0.0
    eranks: list[list[float]] = []
    steps_run = 0
    for step in range(1, num_steps):
        data = policy.build_graph(env, params)
        stats = degree_stats(data.edge_index, num_robots)
        degree_sum += stats.mean
        degree_sq_sum += stats.std**2 + stats.mean**2
        isolated_sum += stats.isolated_fraction
        actions, converged, layer_outputs = policy.act(data)
        if record_rank:
            eranks.append([effective_rank(h) for h in layer_outputs])
        env.StepActions(actions)
        costs[step] = env.GetObjectiveValue() / initial_cost
        steps_run += 1
        if converged:
            _fill_after_convergence(costs, step, costs[step])
            record.converged_step = step
            break
    record.degree_mean = degree_sum / steps_run
    record.degree_std = math.sqrt(max(degree_sq_sum / steps_run - record.degree_mean**2, 0.0))
    record.isolated_fraction = isolated_sum / steps_run
    if record_rank and eranks:
        eranks_array = np.asarray(eranks)
        record.layer_eranks = eranks_array.mean(axis=0)
        record.layer_eranks_std = eranks_array.std(axis=0)
    return record

In [ ]:
%%writefile lpac_study/sweep.py
"""Communication-range sweep of the published LPAC model and the CVT baselines.

Example (full study, as reported):

    python -m lpac_study.sweep --data-dir /path/to/lpac --out-dir results

The script is resumable: rows already present in ``results.csv`` are skipped.
Outputs (all in ``--out-dir``):

* ``results.csv``   one row per (environment, range, robots, controller);
* ``rank.csv``      effective rank per GNN layer for the LPAC rows;
* ``curves.csv``    sub-sampled cost curves, for time-series figures;
* ``run_info.json`` versions, device and timing of the run.
"""
from __future__ import annotations

import argparse
import csv
import json
import platform
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import torch
import torch_geometric
import coverage_control as cc

from .episode import CONTROLLERS, LPAC_NAME, EpisodeRecord, run_cvt_episode, run_lpac_episode
from .lpac_policy import LPACPolicy
from .paths import DataPaths, resolve_data_dir
from .simulation import load_initial_positions, make_environment, make_parameters

DEFAULT_RANGES = (128, 256, 512, 768, 1024)
# (communication range, number of robots) pairs keeping the expected degree
# roughly constant: the number of neighbours within range scales like N * r^2.
DEFAULT_CONSTANT_DEGREE = ("256:32", "512:11", "768:7", "1024:6")
# These two baselines never read the communication range (checked in the C++ source).
RANGE_INDEPENDENT = ("CentralizedCVT", "ClairvoyantCVT")

RESULT_FIELDS = [
    "env", "comm_range", "num_robots", "controller", "sweep",
    "cost_mean", "cost_final", "ratio_mean", "ratio_final",
    "degree_mean", "degree_std", "isolated_fraction", "converged_step",
    "wall_s", "computed",
]
RANK_FIELDS = ["env", "comm_range", "num_robots", "layer", "erank_mean", "erank_std"]
CURVE_FIELDS = ["env", "comm_range", "num_robots", "controller", "step", "cost"]


@dataclass(frozen=True)
class Configuration:
    """One point of a sweep."""

    comm_range: float
    num_robots: int
    sweep: str  # "range" or "constant_degree"


def parse_pair(value: str) -> tuple[int, int]:
    """Parse one ``range:robots`` command-line pair."""
    comm_range, num_robots = value.split(":")
    return int(comm_range), int(num_robots)


def build_configurations(args: argparse.Namespace) -> list[Configuration]:
    """Range sweep at fixed team size, then the constant-degree sweep."""
    configs = [Configuration(float(r), args.num_robots, "range") for r in args.ranges]
    if not args.no_constant_degree:
        pairs = [parse_pair(p) for p in args.constant_degree]
        configs += [Configuration(float(r), n, "constant_degree") for r, n in pairs]
    return configs


def select_device(name: str) -> torch.device:
    """``auto`` picks CUDA when available, else CPU."""
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    return torch.device(name)


@dataclass(frozen=True)
class CostSummary:
    """The two numbers of a clairvoyant run needed to normalize the other controllers."""

    cost_mean: float
    cost_final: float


class ResultWriter:
    """Append-only CSV writers that make the sweep resumable."""

    def __init__(self, out_dir: Path):
        out_dir.mkdir(parents=True, exist_ok=True)
        self.results = out_dir / "results.csv"
        self.rank = out_dir / "rank.csv"
        self.curves = out_dir / "curves.csv"
        self.done = self._read_done_keys()

    def _read_done_keys(self) -> set[tuple]:
        """Keys of rows already in results.csv; clairvoyant costs are kept for normalization."""
        self.clairvoyant: dict[tuple, CostSummary] = {}
        if not self.results.is_file():
            return set()
        done = set()
        with self.results.open(newline="") as handle:
            for r in csv.DictReader(handle):
                key = (int(r["env"]), float(r["comm_range"]), int(r["num_robots"]), r["controller"])
                done.add(key)
                if r["controller"] == "ClairvoyantCVT":
                    self.clairvoyant[(key[0], key[2])] = CostSummary(float(r["cost_mean"]), float(r["cost_final"]))
        return done

    @staticmethod
    def _append(path: Path, fields: list[str], rows: list[dict]) -> None:
        new_file = not path.is_file()
        with path.open("a", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=fields)
            if new_file:
                writer.writeheader()
            writer.writerows(rows)

    def write_result(self, row: dict) -> None:
        self._append(self.results, RESULT_FIELDS, [row])
        self.done.add((row["env"], row["comm_range"], row["num_robots"], row["controller"]))
        if row["controller"] == "ClairvoyantCVT":
            self.clairvoyant[(row["env"], row["num_robots"])] = CostSummary(row["cost_mean"], row["cost_final"])

    def write_rank(self, rows: list[dict]) -> None:
        if rows:
            self._append(self.rank, RANK_FIELDS, rows)

    def write_curve(self, rows: list[dict]) -> None:
        self._append(self.curves, CURVE_FIELDS, rows)


def evaluate_controller(
    controller: str,
    config: Configuration,
    env_index: int,
    paths: DataPaths,
    policy: LPACPolicy | None,
    num_steps: int,
    record_rank: bool,
) -> EpisodeRecord:
    """Build a fresh environment and run one controller on it."""
    params = make_parameters(paths.env_params, config.comm_range, config.num_robots)
    positions = load_initial_positions(paths.pos_file(env_index), config.num_robots)
    env = make_environment(params, paths.env_file(env_index), positions)
    if controller == LPAC_NAME:
        return run_lpac_episode(policy, params, env, num_steps, record_rank)
    return run_cvt_episode(controller, params, env, num_steps)


def result_row(controller, config, env_index, record, clairvoyant, wall_s, computed) -> dict:
    """One line of ``results.csv``; ratios are relative to the clairvoyant run."""
    return {
        "env": env_index, "comm_range": config.comm_range, "num_robots": config.num_robots,
        "controller": controller, "sweep": config.sweep,
        "cost_mean": record.cost_mean, "cost_final": record.cost_final,
        "ratio_mean": record.cost_mean / clairvoyant.cost_mean,
        "ratio_final": record.cost_final / clairvoyant.cost_final,
        "degree_mean": record.degree_mean, "degree_std": record.degree_std,
        "isolated_fraction": record.isolated_fraction,
        "converged_step": "" if record.converged_step is None else record.converged_step,
        "wall_s": round(wall_s, 2), "computed": computed,
    }


def rank_rows(config, env_index, record) -> list[dict]:
    """Effective rank per GNN layer, averaged over the steps of the episode."""
    if record.layer_eranks is None:
        return []
    return [
        {"env": env_index, "comm_range": config.comm_range, "num_robots": config.num_robots,
         "layer": layer, "erank_mean": float(mean), "erank_std": float(std)}
        for layer, (mean, std) in enumerate(zip(record.layer_eranks, record.layer_eranks_std))
    ]


def curve_rows(controller, config, env_index, record, stride) -> list[dict]:
    """Cost curve sub-sampled every ``stride`` steps (the last step is always kept)."""
    steps = list(range(0, len(record.costs), stride))
    if steps[-1] != len(record.costs) - 1:
        steps.append(len(record.costs) - 1)
    return [
        {"env": env_index, "comm_range": config.comm_range, "num_robots": config.num_robots,
         "controller": controller, "step": s, "cost": float(record.costs[s])}
        for s in steps
    ]


def run_sweep(args: argparse.Namespace) -> None:
    """Main loop: configurations x environments x controllers, written as they finish."""
    paths = resolve_data_dir(args.data_dir)
    device = select_device(args.device)
    torch.manual_seed(args.seed)
    out_dir = Path(args.out_dir)
    writer = ResultWriter(out_dir)
    policy = LPACPolicy(paths, device, args.model, args.variant) if LPAC_NAME in args.controllers else None
    num_envs = min(args.num_envs, paths.count_environments())
    env_indices = range(args.env_start, num_envs if args.env_end is None else min(args.env_end, num_envs))
    cache: dict[tuple, EpisodeRecord] = {}  # (env, robots, controller) of range-independent CVTs
    started = time.time()
    print(f"device={device} envs={num_envs} steps={args.num_steps} out={out_dir}", flush=True)

    # Clairvoyant first: every other controller is normalized by it.
    order = sorted(args.controllers, key=lambda c: c != "ClairvoyantCVT")
    for config in build_configurations(args):
        for env_index in env_indices:
            for controller in order:
                key = (env_index, config.comm_range, config.num_robots, controller)
                if key in writer.done:
                    continue
                cache_key = (env_index, config.num_robots, controller)
                reuse = args.reuse_range_independent and controller in RANGE_INDEPENDENT
                tic = time.time()
                if reuse and cache_key in cache:
                    record, computed = cache[cache_key], False
                else:
                    record = evaluate_controller(
                        controller, config, env_index, paths, policy, args.num_steps, not args.no_rank
                    )
                    cache[cache_key] = record
                    computed = True
                wall = time.time() - tic
                clairvoyant = cache.get((env_index, config.num_robots, "ClairvoyantCVT"))                     or writer.clairvoyant.get((env_index, config.num_robots))
                if clairvoyant is None:
                    raise RuntimeError("ClairvoyantCVT must be evaluated first to normalize costs")
                writer.write_result(result_row(controller, config, env_index, record, clairvoyant, wall, computed))
                writer.write_curve(curve_rows(controller, config, env_index, record, args.curve_stride))
                if controller == LPAC_NAME:
                    writer.write_rank(rank_rows(config, env_index, record))
                print(
                    f"[{time.time() - started:7.0f}s] r={config.comm_range:6.0f} N={config.num_robots:2d} "
                    f"env={env_index:2d} {controller:16s} cost_mean={record.cost_mean:.4f} "
                    f"ratio={record.cost_mean / clairvoyant.cost_mean:.3f} deg={record.degree_mean:5.2f} "
                    f"({wall:5.1f}s{'' if computed else ', reused'})",
                    flush=True,
                )
    write_run_info(out_dir, args, device, time.time() - started)


def write_run_info(out_dir: Path, args: argparse.Namespace, device: torch.device, elapsed: float) -> None:
    """Record software versions and timing next to the results."""
    info = {
        "args": vars(args), "device": str(device), "elapsed_s": round(elapsed, 1), "model": args.model,
        "variant": args.variant,
        "python": sys.version.split()[0], "platform": platform.platform(),
        "torch": torch.__version__, "torch_geometric": torch_geometric.__version__,
        "coverage_control": getattr(cc, "__version__", "unknown"),
    }
    (out_dir / "run_info.json").write_text(json.dumps(info, indent=2))


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter
    )
    parser.add_argument("--data-dir", help="extracted 'lpac' folder (or set $LPAC_DATA_DIR)")
    parser.add_argument("--out-dir", default="results", help="where CSV outputs are written")
    parser.add_argument("--ranges", type=float, nargs="*", default=DEFAULT_RANGES,
                        help="communication ranges in metres for the range sweep (empty list skips it)")
    parser.add_argument("--num-robots", type=int, default=32, help="team size of the range sweep")
    parser.add_argument("--constant-degree", nargs="+", default=DEFAULT_CONSTANT_DEGREE,
                        metavar="RANGE:ROBOTS", help="(range, robots) pairs of the constant-degree sweep")
    parser.add_argument("--no-constant-degree", action="store_true", help="skip the constant-degree sweep")
    parser.add_argument("--num-envs", type=int, default=11, help="number of release environments to use")
    parser.add_argument("--num-steps", type=int, default=600, help="episode length (NumSteps of eval.toml)")
    parser.add_argument("--env-start", type=int, default=0, help="first environment index (to split work)")
    parser.add_argument("--env-end", type=int, default=None, help="one past the last environment index")
    parser.add_argument("--controllers", nargs="+", default=list(CONTROLLERS), choices=CONTROLLERS)
    parser.add_argument("--reuse-range-independent", action="store_true",
                        help="run Centralized/Clairvoyant CVT once per (env, robots) instead of once per range")
    parser.add_argument("--no-rank", action="store_true", help="skip the effective-rank measurement")
    parser.add_argument("--curve-stride", type=int, default=10, help="keep every k-th step of the cost curves")
    parser.add_argument("--device", default="auto", help="auto, cpu or cuda")
    parser.add_argument("--seed", type=int, default=0, help="torch seed (the pipeline is deterministic anyway)")
    parser.add_argument("--model", default=None, help="state dict of another LPAC-K3 checkpoint (default: the published one)")
    parser.add_argument("--variant", default="V0", choices=("V0", "V1", "V2", "V3", "V0R", "V4", "V5", "V6", "V2R", "V2RL", "V2B", "V2M"), help="architecture of --model (lpac_study.variants)")
    return parser


def main(argv: list[str] | None = None) -> None:
    run_sweep(build_parser().parse_args(argv))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile lpac_study/factorial.py
"""Inference-time factorial experiment: which input carries the neighbourhood that breaks LPAC?

    python -m lpac_study.factorial --data-dir /path/to/lpac --baseline results/results.csv \
        --out-dir results_2x2 --ranges 1024 --cells G0C0 G0C1 G0C2 G1C0 G1C1 G1C2 --k 4

Factor GRAPH (what the GNN sees through ``edge_index``):
  G0  every edge within range (the published behaviour);
  G1  each robot keeps its k nearest robots among those in range, made undirected;
  R<m> edges only between robots within min(<m>, test range) metres: the graph analogue of
       the fixed-scale map (R256 = edges capped at 256 m), for the map-less variants.
Factor MAP (what the CNN sees through the two communication-map channels):
  C0  the library's map (all robots in range, resolution of the test range);
  C1  only the k nearest neighbours of G1 (same undirected set), test-range resolution;
  C2  only robots within min(128 m, test range), drawn with the 128 m normalization and resolution;
  C3  only robots within min(256 m, test range), drawn with the 256 m normalization and resolution;
  C4  empty map: both neighbour channels set to zero (the library's value for "no neighbour");
  K<m> every robot within the test range drawn at the <m> metre scale, those farther than <m>
       projected onto the <m> circle (border cell, value of norm 1) instead of dropped: the
       "clamp" answer to the TA's question, against S<m>/C3 which drop them;
  S<m> map of the robots within min(<m>, test range) metres, drawn with the <m> m normalization
       and resolution (C2 = S128, C3 = S256); only robots within radio range are ever drawn.
``--model`` evaluates another LPAC-K3 checkpoint (same architecture); ``--model-tag`` labels it.
The simulator itself always runs at the test range; only the model inputs change.
Costs are normalized by the clairvoyant rows of ``--baseline`` (range-independent).

Sensitivity repeats (``--repeats N --noise-sigma 1``): the initial positions are shifted
by Gaussian noise with a fixed seed per (environment, repeat), and the clairvoyant is
re-run on the same shifted start to normalize.  ``repeat = 0`` is the nominal episode.

Outputs ``results_2x2.csv`` (one row per environment, range, cell, k, repeat) in
``--out-dir``; the script is resumable.
"""
from __future__ import annotations

import argparse
import csv
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data

from .episode import run_cvt_episode, run_lpac_episode
from .lpac_policy import CNN_MAP_SIZE, LPACPolicy
from .neighbourhood import (comm_maps_from_positions, in_range_mask, knn_within_range_mask, mask_to_edge_index,
                            robot_positions_double, symmetrize)
from .paths import DataPaths, resolve_data_dir
from .simulation import load_initial_positions, make_environment, make_parameters

NUM_ROBOTS = 32
COMM_CHANNELS = [1, 2]
MAP_SCALE_LEVELS = {"C2": 128.0, "C3": 256.0}  # map drawn as if the range were this value
CELLS = tuple(f"G{g}C{c}" for g in (0, 1) for c in range(5))
FIELDS = ["env", "comm_range", "model", "graph", "map", "cell", "k", "repeat", "noise_sigma",
          "cost_mean", "cost_final", "ratio_mean", "ratio_final", "clairvoyant_cost_mean",
          "degree_mean", "degree_std", "map_neighbours_mean", "converged_step", "wall_s"]
WORLD_MARGIN = 0.5  # metres kept from the world boundary when perturbing initial positions


class FactorialPolicy(LPACPolicy):
    """The published policy whose graph and communication maps are rebuilt per factor level."""

    def __init__(self, paths, device, graph: str, comm_map: str, k: int, state_dict=None, variant: str = "V0"):
        super().__init__(paths, device, state_dict, variant)
        self.graph, self.comm_map, self.k = graph, comm_map, k
        self.map_neighbour_counts: list[float] = []

    def build_graph(self, env, params) -> Data:
        data = super().build_graph(env, params)
        positions = robot_positions_double(env)
        test_range = params.pCommunicationRange
        capped = symmetrize(knn_within_range_mask(positions, test_range, self.k))
        if self.graph == "G1":
            data.edge_index = mask_to_edge_index(capped)
        elif graph_scale(self.graph) is not None:
            data.edge_index = mask_to_edge_index(in_range_mask(positions, min(graph_scale(self.graph), test_range)))
        map_mask = self._map_mask(positions, test_range, capped)
        if self.comm_map == "C4":
            data.x[:, COMM_CHANNELS] = 0.0
        elif self.comm_map != "C0":
            map_range = map_scale(self.comm_map) or test_range
            data.x[:, COMM_CHANNELS] = comm_maps_from_positions(positions, map_mask, map_range, CNN_MAP_SIZE,
                                                                clamp=clamps_far_robots(self.comm_map))
        self.map_neighbour_counts.append(float(map_mask.sum(1).float().mean()))
        return data

    def _map_mask(self, positions, test_range, capped):
        """Robots drawn on each robot's communication map for the current MAP level."""
        if self.comm_map == "C1":
            return capped
        scale = map_scale(self.comm_map)
        if clamps_far_robots(self.comm_map):
            # K<m>: every robot the radio reaches is drawn, the far ones on the border cell.
            return in_range_mask(positions, test_range)
        if scale is not None:
            # A robot can only draw neighbours it can hear: within the test range, and within
            # the drawing scale so that they fit on the map (corrected on 14 October; before,
            # a scale larger than the test range drew robots out of radio range).
            return in_range_mask(positions, min(scale, test_range))
        if self.comm_map == "C4":
            return torch.zeros_like(capped)
        return in_range_mask(positions, test_range)


def map_scale(level: str):
    """Fixed drawing scale (metres) of a MAP level: C2 = 128, C3 = 256, S<m>/K<m> = any value, else None."""
    if level in MAP_SCALE_LEVELS:
        return MAP_SCALE_LEVELS[level]
    if level[:1] in ("S", "K") and level[1:].isdigit():
        return float(level[1:])
    return None


def clamps_far_robots(level: str) -> bool:
    """Whether a MAP level keeps the robots beyond its scale by projecting them on the border (K<m>)."""
    return level.startswith("K") and level[1:].isdigit()


def graph_scale(level: str):
    """Edge cap (metres) of a GRAPH level R<metres>, else None."""
    if level.startswith("R") and level[1:].isdigit():
        return float(level[1:])
    return None


def parse_cell(cell: str):
    """Split a cell name such as G0C3, G1S192 or R256C0 into its GRAPH and MAP levels."""
    match = re.fullmatch(r"(G[01]|R\d+)([CSK]\d+)", cell)
    if match is None:
        raise ValueError("bad cell name %r: expected G<0|1> or R<metres>, followed by C<0-4>, S<metres> or K<metres>" % cell)
    return match.group(1), match.group(2)


def load_clairvoyant(baseline: Path, num_robots: int) -> dict[int, tuple[float, float]]:
    """Per-environment clairvoyant (cost_mean, cost_final) from an existing results.csv."""
    rows = pd.read_csv(baseline)
    rows = rows[(rows["controller"] == "ClairvoyantCVT") & (rows["num_robots"] == num_robots)]
    rows = rows.drop_duplicates("env")
    return {int(r.env): (float(r.cost_mean), float(r.cost_final)) for r in rows.itertuples()}


def done_keys(path: Path) -> set[tuple]:
    """Keys (env, range, cell, k, repeat) already present in a results file."""
    if not path.is_file():
        return set()
    with path.open(newline="") as handle:
        return {(int(r["env"]), float(r["comm_range"]), r.get("model") or "published", r["cell"], int(r["k"]),
                 int(r.get("repeat") or 0)) for r in csv.DictReader(handle)}


def append_row(path: Path, row: dict) -> None:
    new = not path.is_file()
    with path.open("a", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=FIELDS)
        if new:
            writer.writeheader()
        writer.writerow(row)


def perturbed_positions(positions: np.ndarray, env_index: int, repeat: int, sigma: float, world_size: float) -> np.ndarray:
    """Initial positions shifted by N(0, sigma) metres with a fixed seed; repeat 0 is unchanged."""
    if repeat == 0 or sigma <= 0:
        return positions
    rng = np.random.default_rng(1000 * env_index + repeat)
    shifted = positions + rng.normal(0.0, sigma, size=positions.shape)
    return np.clip(shifted, WORLD_MARGIN, world_size - WORLD_MARGIN)


def run_cell(paths: DataPaths, env_index: int, comm_range: float, cell: str, k: int, repeat: int,
             sigma: float, num_steps: int, clairvoyant: dict, device, model=None, model_tag="published",
             variant="V0") -> dict:
    """One LPAC episode for one factorial cell on one (possibly perturbed) environment."""
    graph, comm_map = parse_cell(cell)
    policy = FactorialPolicy(paths, device, graph, comm_map, k, model, variant)
    params = make_parameters(paths.env_params, comm_range, NUM_ROBOTS)
    positions = load_initial_positions(paths.pos_file(env_index), NUM_ROBOTS)
    positions = perturbed_positions(positions, env_index, repeat, sigma, params.pWorldMapSize)
    if repeat == 0:
        clair_mean, clair_final = clairvoyant[env_index]
    else:  # the shifted start changes the reference too: re-run the clairvoyant on it
        reference = run_cvt_episode("ClairvoyantCVT", params, make_environment(params, paths.env_file(env_index), positions), num_steps)
        clair_mean, clair_final = reference.cost_mean, reference.cost_final
    env = make_environment(params, paths.env_file(env_index), positions)
    tic = time.time()
    record = run_lpac_episode(policy, params, env, num_steps, record_rank=False)
    return {
        "env": env_index, "comm_range": comm_range, "model": model_tag, "graph": graph, "map": comm_map,
        "cell": cell, "k": k,
        "repeat": repeat, "noise_sigma": sigma if repeat else 0.0,
        "cost_mean": record.cost_mean, "cost_final": record.cost_final,
        "ratio_mean": record.cost_mean / clair_mean, "ratio_final": record.cost_final / clair_final,
        "clairvoyant_cost_mean": clair_mean,
        "degree_mean": record.degree_mean, "degree_std": record.degree_std,
        "map_neighbours_mean": sum(policy.map_neighbour_counts) / len(policy.map_neighbour_counts),
        "converged_step": "" if record.converged_step is None else record.converged_step,
        "wall_s": round(time.time() - tic, 1),
    }


def run_factorial(args: argparse.Namespace) -> None:
    paths = resolve_data_dir(args.data_dir)
    device = torch.device("cpu")
    torch.manual_seed(0)
    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    results = out / "results_2x2.csv"
    done = done_keys(results)
    clairvoyant = load_clairvoyant(Path(args.baseline), NUM_ROBOTS)
    env_end = args.num_envs if args.env_end is None else min(args.env_end, args.num_envs)
    repeats = range(args.repeat_start, args.repeats + 1)
    started = time.time()
    for comm_range in args.ranges:
        for env_index in range(args.env_start, env_end):
            for cell in args.cells:
                for k in args.k:
                    graph, comm_map = parse_cell(cell)
                    if comm_map != "C1" and graph != "G1" and k != args.k[0]:
                        continue  # these cells do not depend on k: computed once
                    for repeat in repeats:
                        key = (env_index, float(comm_range), args.model_tag, cell, int(k), repeat)
                        if key in done:
                            continue
                        row = run_cell(paths, env_index, float(comm_range), cell, int(k), repeat,
                                       args.noise_sigma, args.num_steps, clairvoyant, device,
                                       args.model, args.model_tag, args.variant)
                        append_row(results, row)
                        done.add(key)
                        print(f"[{time.time() - started:6.0f}s] r={comm_range:5.0f} env={env_index:2d} "
                              f"{args.model_tag} {cell} k={k:2d} "
                              f"rep={repeat} ratio={row['ratio_mean']:.3f} deg={row['degree_mean']:5.2f} "
                              f"map_nb={row['map_neighbours_mean']:5.2f} ({row['wall_s']:.0f}s)", flush=True)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir")
    parser.add_argument("--baseline", required=True, help="results.csv of the range sweep (clairvoyant normalization)")
    parser.add_argument("--out-dir", default="results_2x2")
    parser.add_argument("--ranges", type=float, nargs="+", default=[1024.0])
    parser.add_argument("--cells", nargs="+", default=list(CELLS[:3]) + list(CELLS[5:8]),
                        help="cells G<0|1><level>, level in C0..C4 or S<metres> (map drawn at that fixed scale)")
    parser.add_argument("--model", default=None, help="state dict of another LPAC-K3 checkpoint (default: published)")
    parser.add_argument("--model-tag", default="published", help="label of the checkpoint written in the CSV")
    parser.add_argument("--variant", default="V0", choices=("V0", "V1", "V2", "V3", "V0R"), help="architecture of --model")
    parser.add_argument("--k", type=int, nargs="+", default=[4], help="neighbour caps for G1/C1")
    parser.add_argument("--num-envs", type=int, default=11)
    parser.add_argument("--env-start", type=int, default=0)
    parser.add_argument("--env-end", type=int, default=None)
    parser.add_argument("--num-steps", type=int, default=600)
    parser.add_argument("--repeats", type=int, default=0, help="number of perturbed repeats per episode (0 = nominal only)")
    parser.add_argument("--repeat-start", type=int, default=0, help="first repeat index to run (1 skips the nominal episode)")
    parser.add_argument("--noise-sigma", type=float, default=1.0, help="std (m) of the initial-position noise of repeats")
    return parser


def main(argv: list[str] | None = None) -> None:
    run_factorial(build_parser().parse_args(argv))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile lpac_study/merge.py
"""Merge the CSV outputs of several sweep (or factorial) processes and print a summary table.

    python -m lpac_study.merge --parts results/part_a results/part_b --out-dir results
    python -m lpac_study.merge --kind factorial --parts results_2x2/f1024a results_2x2/r768_env_0 --out-dir results_2x2

Rows are de-duplicated on (env, comm_range, num_robots, controller); the
first occurrence wins.  The summary printed at the end is the mean and the
standard deviation across environments of the clairvoyant-normalized cost.
"""
from __future__ import annotations

import argparse
from pathlib import Path

import pandas as pd

KEY = ["env", "comm_range", "num_robots", "controller"]
FILES = {"results.csv": KEY, "rank.csv": ["env", "comm_range", "num_robots", "layer"],
         "curves.csv": KEY + ["step"]}
FACTORIAL_KEY = ["env", "comm_range", "model", "cell", "k", "repeat"]
FACTORIAL_FILES = {"results_2x2.csv": FACTORIAL_KEY}


def merge_file(parts: list[Path], name: str, key: list[str]) -> pd.DataFrame:
    """Concatenate one CSV across parts, dropping duplicate keys."""
    frames = [pd.read_csv(p / name) for p in parts if (p / name).is_file()]
    if not frames:
        return pd.DataFrame()
    merged = pd.concat(frames, ignore_index=True)
    if "repeat" in key:  # files written before the sensitivity repeats existed have no such column
        merged["repeat"] = merged.get("repeat", 0).fillna(0).astype(int)
        merged["noise_sigma"] = merged.get("noise_sigma", 0.0).fillna(0.0)
    if "model" in key:  # files written before --model-tag existed hold the published checkpoint
        merged["model"] = merged.get("model", "published").fillna("published")
    return merged.drop_duplicates(subset=key).sort_values(key)


def factorial_summary(results: pd.DataFrame, metric: str = "ratio_mean") -> pd.DataFrame:
    """Mean ± std across environments per (range, cell, k) of the factorial experiment."""
    nominal = results[results["repeat"] == 0] if "repeat" in results else results
    grouped = nominal.groupby(["comm_range", "cell", "k"])
    table = grouped[metric].agg(mean="mean", std="std", n_env="count").reset_index()
    extras = grouped[["degree_mean", "map_neighbours_mean"]].mean().reset_index()
    return table.merge(extras).round(3)


def summary(results: pd.DataFrame, metric: str = "ratio_mean") -> pd.DataFrame:
    """Mean ± std across environments, one row per (sweep, range, robots, controller)."""
    grouped = results.groupby(["sweep", "comm_range", "num_robots", "controller"])[metric]
    table = grouped.agg(mean="mean", std="std", n_env="count").reset_index()
    degrees = results[results["controller"] == "lpac"].groupby(
        ["sweep", "comm_range", "num_robots"])["degree_mean"].mean().rename("degree_lpac")
    return table.merge(degrees.reset_index(), how="left").round(4)


def main(argv: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--parts", nargs="+", required=True, help="output folders of sweep processes")
    parser.add_argument("--out-dir", required=True, help="folder receiving the merged CSVs")
    parser.add_argument("--kind", choices=["sweep", "factorial"], default="sweep",
                        help="sweep: results/rank/curves of lpac_study.sweep; factorial: results_2x2.csv")
    args = parser.parse_args(argv)
    parts = [Path(p) for p in args.parts]
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    files = FILES if args.kind == "sweep" else FACTORIAL_FILES
    for name, key in files.items():
        merged = merge_file(parts, name, key)
        if not merged.empty:
            merged.to_csv(out_dir / name, index=False)
            print(f"{name}: {len(merged)} rows")
    with pd.option_context("display.width", 200, "display.max_rows", 500):
        if args.kind == "sweep":
            print(summary(pd.read_csv(out_dir / "results.csv")).to_string(index=False))
        else:
            print(factorial_summary(pd.read_csv(out_dir / "results_2x2.csv")).to_string(index=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile lpac_study/checks.py
"""Mandatory checks before the factorial experiment: the rebuilt inputs match the library.

    python -m lpac_study.checks --data-dir /path/to/lpac [--comm-range 1024] [--steps 20]

Runs the published policy for a few steps at the given range and, at every
step, compares against ``CoverageEnvUtils.get_torch_geometric_data``:

1. the communication-map channels rebuilt from positions with all in-range
   neighbours (maximum absolute error reported);
2. which channels of ``data.x`` are communication maps (1 and 2) and which are
   perception maps (0: local importance map, 3: obstacle map), by checking
   that only channels 1 and 2 change when neighbours are removed;
3. the library's ``edge_index`` equals the in-range graph, and the simulator's
   own neighbour count matches it (the simulated range is untouched).
"""
from __future__ import annotations

import argparse

import torch
from coverage_control import CoverageEnvUtils
from torch_geometric.utils import to_undirected

from .lpac_policy import CNN_MAP_SIZE, LPACPolicy
from .neighbourhood import (comm_maps_from_positions, in_range_mask, knn_within_range_mask, mask_to_edge_index,
                            robot_positions_double, symmetrize)
from .paths import resolve_data_dir
from .simulation import load_initial_positions, make_environment, make_parameters

COMM_CHANNELS = (1, 2)


def edge_set(edge_index: torch.Tensor) -> set[tuple[int, int]]:
    return set(map(tuple, edge_index.t().tolist()))


def run_checks(args: argparse.Namespace) -> None:
    paths = resolve_data_dir(args.data_dir)
    params = make_parameters(paths.env_params, args.comm_range, args.num_robots)
    env = make_environment(params, paths.env_file(args.env), load_initial_positions(paths.pos_file(args.env), args.num_robots))
    policy = LPACPolicy(paths, torch.device("cpu"))
    worst_map_error = 0.0
    worst_other_change = 0.0
    max_lonely = 0
    for step in range(args.steps):
        data = policy.build_graph(env, params)
        positions = robot_positions_double(env)
        full_mask = in_range_mask(positions, params.pCommunicationRange)
        rebuilt = comm_maps_from_positions(positions, full_mask, params.pCommunicationRange, CNN_MAP_SIZE)
        worst_map_error = max(worst_map_error, float((data.x[:, list(COMM_CHANNELS)] - rebuilt).abs().max()))
        # Channels that must not depend on neighbours: compare with a restricted neighbour set.
        restricted = comm_maps_from_positions(positions, symmetrize(knn_within_range_mask(positions, params.pCommunicationRange, 2)),
                                              params.pCommunicationRange, CNN_MAP_SIZE)
        assert not torch.equal(restricted, rebuilt) or full_mask.sum() <= 2 * len(positions)
        other = [c for c in range(data.x.shape[1]) if c not in COMM_CHANNELS]
        worst_other_change = max(worst_other_change, float(data.x[:, other].abs().sum() == 0))
        # Graph: library edge_index (d <= range) versus strict in-range mask.
        library_edges = edge_set(to_undirected(data.edge_index))
        rebuilt_edges = edge_set(mask_to_edge_index(full_mask))
        if library_edges != rebuilt_edges:
            raise AssertionError(f"step {step}: edge sets differ ({len(library_edges ^ rebuilt_edges)} edges)")
        simulator_counts = [len(env.GetRelativePositonsNeighbors(i)) for i in range(len(positions))]
        if simulator_counts != full_mask.sum(1).tolist():
            raise AssertionError(f"step {step}: simulator neighbour counts differ from the in-range mask")
        # A robot without neighbour must have all-zero communication channels in the library's map.
        lonely = [i for i, n in enumerate(simulator_counts) if n == 0]
        if lonely and float(data.x[lonely][:, list(COMM_CHANNELS)].abs().max()) != 0.0:
            raise AssertionError(f"step {step}: a robot without neighbour has a non-zero communication map")
        max_lonely = max(max_lonely, len(lonely))
        actions, _, _ = policy.act(data)
        env.StepActions(actions)
    print(f"steps checked: {args.steps} at {args.comm_range:.0f} m, {args.num_robots} robots, env {args.env}")
    print(f"comm-map reconstruction, max |error| over all steps: {worst_map_error:.3e}")
    print(f"x channels: {data.x.shape[1]} (comm maps = {COMM_CHANNELS}; others = perception maps, never zero: "
          f"{'ok' if worst_other_change == 0 else 'EMPTY'})")
    print(f"edge_index equals the in-range graph and the simulator's neighbour lists: ok")
    print(f"robots without neighbour have all-zero map channels: ok "
          f"(up to {max_lonely} such robots in one step)")
    print(f"simulated communication range after the checks: {params.pCommunicationRange:.0f} m")


def main(argv: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir")
    parser.add_argument("--comm-range", type=float, default=1024.0)
    parser.add_argument("--num-robots", type=int, default=32)
    parser.add_argument("--env", type=int, default=0)
    parser.add_argument("--steps", type=int, default=20)
    run_checks(parser.parse_args(argv))


if __name__ == "__main__":
    main()

In [ ]:
import subprocess, sys, time
import pandas as pd

def run_parallel(make_cmd, envs, label):
    """Run one process per environment, WORKERS at a time; make_cmd(env) returns the argument list."""
    t0, pending, running = time.time(), list(envs), []
    while pending or running:
        while pending and len(running) < WORKERS:
            env = pending.pop(0)
            log = open(OUT / f"{label}_env{env}.log", "w")
            running.append((env, subprocess.Popen(make_cmd(env), stdout=log, stderr=subprocess.STDOUT), log))
        time.sleep(5)
        for item in list(running):
            env, proc, log = item
            if proc.poll() is not None:
                log.close(); running.remove(item)
                if proc.returncode != 0:
                    print(open(OUT / f"{label}_env{env}.log").read()[-3000:])
                    raise RuntimeError(f"{label}: environment {env} failed")
                print(f"{label}: environment {env} done ({time.time() - t0:.0f} s)")

def env_args(env):
    return ["--num-envs", str(NUM_ENVS), "--env-start", str(env), "--env-end", str(env + 1), "--num-steps", str(NUM_STEPS)]

def merge(parts, out_dir, kind="sweep"):
    subprocess.run([sys.executable, "-m", "lpac_study.merge", "--kind", kind, "--parts", *map(str, parts),
                    "--out-dir", str(out_dir)], check=True, capture_output=True)

ENVS = list(range(NUM_ENVS))

In [ ]:
# Sanity check: the neighbour map and graph rebuilt by lpac_study match the library's own, bit for bit.
subprocess.run([sys.executable, "-m", "lpac_study.checks", "--comm-range", "1024", "--steps", "5"], check=True)

## 1. The anomaly: more range, worse coverage

The published checkpoint was trained with a 128 m range. We run it, and the clairvoyant
controller used for normalization, at 256, 512 and 1024 m. On the 40-environment set of the post
the costs are 1.70, 2.87 and 6.15.

In [ ]:
SWEEP = OUT / "sweep"
run_parallel(lambda env: [sys.executable, "-m", "lpac_study.sweep", "--out-dir", str(SWEEP / f"env_{env}"),
                          "--ranges", "256", "512", "1024", "--controllers", "lpac", "ClairvoyantCVT",
                          "--no-constant-degree", "--no-rank", *env_args(env)], ENVS, "sweep")
merge([SWEEP / f"env_{e}" for e in ENVS], SWEEP)
sweep = pd.read_csv(SWEEP / "results.csv")
published = sweep[sweep.controller == "lpac"].pivot_table(index="env", columns="comm_range", values="ratio_mean")
published.loc["mean"] = published.mean()
published.round(2)

## 2. Graph or picture of the graph? A factorial at 1024 m

The neighbourhood enters LPAC twice: as the `edge_index` of the GNN (**G0** every robot in range,
**G1** only the 4 nearest) and as the two neighbour-map channels of the CNN (**C0** as published,
drawn at the scale of the range; **C3** drawn at a fixed 256 m scale, robots beyond it left out;
**C4** empty). Same weights, same simulator; only the model's inputs change.

In [ ]:
FACT = OUT / "factorial"
def factorial(tag, ranges, cells):
    run_parallel(lambda env: [sys.executable, "-m", "lpac_study.factorial", "--baseline", str(SWEEP / "results.csv"),
                              "--out-dir", str(FACT / f"{tag}_env_{env}"), "--ranges", *ranges, "--k", "4",
                              "--cells", *cells, *env_args(env)], ENVS, f"factorial_{tag}")
factorial("r1024", ["1024"], ["G1C0", "G0C3", "G0C4"])   # the factorial proper
factorial("r512", ["512"], ["G0C3"])                      # the fix at 512 m, for part 3
merge([FACT / f"{t}_env_{e}" for t in ("r1024", "r512") for e in ENVS], FACT, kind="factorial")
fact = pd.read_csv(FACT / "results_2x2.csv")
fact = fact[fact.repeat == 0]
asis = sweep[sweep.controller == "lpac"].assign(cell="G0C0")[["env", "comm_range", "cell", "ratio_mean"]]
cells = pd.concat([asis, fact[["env", "comm_range", "cell", "ratio_mean"]]])
NAMES = {"G0C0": "as published", "G1C0": "graph capped to 4 nearest", "G0C4": "empty map",
         "G0C3": "map at fixed 256 m scale"}
table_1024 = (cells[cells.comm_range == 1024].groupby("cell").ratio_mean.mean()
              .rename(index=NAMES).sort_values(ascending=False))
table_1024.round(2)

## 3. The fix: a fixed 256 m map, no retraining

Drawing the neighbour map at a fixed 256 m scale, whatever the radio range, brings the published
model back to its own 256 m optimum at every range (at 256 m the fixed map *is* the published map).

In [ ]:
import matplotlib.pyplot as plt
mean_asis = published.loc["mean"]
fixed = cells[cells.cell == "G0C3"].groupby("comm_range").ratio_mean.mean()
fixed.loc[256] = mean_asis[256]
fixed = fixed.sort_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.plot(mean_asis.index, mean_asis.values, "o-", color="#D55E00", lw=2, label="LPAC as published")
ax.plot(fixed.index, fixed.values, "o-", color="#0072B2", lw=2, label="same weights, fixed 256 m map")
ax.set_xscale("log", base=2); ax.set_xticks([256, 512, 1024]); ax.set_xticklabels(["256", "512", "1024"])
ax.set_xlabel("communication range (m)"); ax.set_ylabel("cost / clairvoyant (lower is better)")
ax.set_title("More communication, worse coverage, and the fix"); ax.legend(); ax.grid(alpha=0.3)
ax = axes[1]
COLOURS = {"as published": "#D55E00", "graph capped to 4 nearest": "#E69F00", "empty map": "#999999",
           "map at fixed 256 m scale": "#0072B2"}
ax.barh(table_1024.index, table_1024.values, color=[COLOURS[n] for n in table_1024.index])
ax.set_xlabel("cost / clairvoyant at 1024 m"); ax.set_title("Factorial at 1024 m: change one input")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.savefig(OUT / "summary.png", dpi=150); plt.show()

## 4. V4: let the graph carry the geometry

V4 removes the two neighbour-map channels and adds one PointNet-style `MessagePassing` layer in
front of the unchanged `TAGConv` stack. On every edge j → i at distance d it reads the unit
direction (p_j − p_i) / d and tanh(d / 256 m), three numbers in [−1, 1] whatever the range, and
aggregates with a max (`lpac_study/variants.py`, class `BoundedGeometryConv`). Trained on mixed
ranges, it is the only model of the study whose cost keeps falling beyond 256 m. The checkpoint
is seed 0 of that recipe; the clairvoyant rows of part 1 are reused.

In [ ]:
if not V4_URL:
    print("V4_URL is empty: part 4 skipped.")
else:
    import urllib.request
    V4 = OUT / "v4"
    ckpt = OUT / "v4_state_dict.pt"
    if not ckpt.is_file():
        urllib.request.urlretrieve(V4_URL, ckpt)
    for env in ENVS:  # seed each folder with the clairvoyant rows so that they are not recomputed
        (V4 / f"env_{env}").mkdir(parents=True, exist_ok=True)
        part = pd.read_csv(SWEEP / f"env_{env}" / "results.csv")
        part[part.controller == "ClairvoyantCVT"].to_csv(V4 / f"env_{env}" / "results.csv", index=False)
    run_parallel(lambda env: [sys.executable, "-m", "lpac_study.sweep", "--out-dir", str(V4 / f"env_{env}"),
                              "--ranges", "256", "512", "1024", "--controllers", "lpac", "ClairvoyantCVT",
                              "--no-constant-degree", "--no-rank", "--model", str(ckpt), "--variant", "V4",
                              *env_args(env)], ENVS, "v4")
    merge([V4 / f"env_{e}" for e in ENVS], V4)
    v4 = pd.read_csv(V4 / "results.csv")
    v4 = v4[v4.controller == "lpac"].groupby("comm_range").ratio_mean.mean()
    display(pd.DataFrame({"LPAC as published": mean_asis, "fixed 256 m map": fixed, "V4 (mixed ranges)": v4}).round(2))

## What next

* The post explains each result and its limits; the 40-environment numbers there are more precise
  than the 2 environments of quick mode.
* The project site has every number of the study, interactive: [https://loicpoisson.github.io/lpac-range-transfer/](https://loicpoisson.github.io/lpac-range-transfer/).
* The last cell zips `{OUT}` for download.

In [ ]:
import shutil
zip_path = shutil.make_archive(str(OUT.parent / "lpac_range_results"), "zip", root_dir=OUT)  # outside OUT
print("archive:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as error:  # not on Colab
    print("download it manually:", error)